# 10年定着予測 - CatBoostネイティブtext_features（69_）

## 位置づけ

`reference/長期定着予測_crmaine_0816.ipynb`（**現1位が過去に使用していたと見られる**ノートブック）を
精読した結果、自分たちが未検証だった軸が見つかった。

**自分たちの現行パイプラインは入社時メモをTF-IDF文字n-gram→SVD 15次元に圧縮している。**
0816版はこれをせず、**CatBoostのネイティブtext_features機構に生テキストを直接渡している**
（janomeで名詞・動詞・形容詞だけ残して空白区切りにしたもの）。text_featuresは学習時に
fold内でリークしない形でBoW/Naive Bayes/BM25統計を内部生成する仕組みで、固定次元への
SVD圧縮とは情報の持ち方が根本的に違う。0816版の特徴量重要度グラフでは
「学習・メモ・残業が上位」と明記されており、text_features化したメモ・自己学習テーマが
実際に効いている可能性がある。

**この軸は`57_`の総当たり探索（multilingual-e5-small埋め込みは試したが、
CatBoostのtext_features機構そのものは未検証）でもカバーされていない。**

## 検証する2つの新規text_features

1. **メモtxt**: 入社時メモをjanomeで形態素解析（名詞・動詞・形容詞のみ）→ 空白区切り
2. **学習txt**: `自己学習（詳細）`の24ヶ月分からコーステーマ名を全て抽出し、空白区切りで連結
   （`59_`のSLブロックは「DX_IT/対人営業...」の離散キーワードバケットに分類する設計だったが、
   ここではテーマ名の生テキストをそのままCatBoostに解釈させる。実装が異なるため
   `59_`の否定結果（Public+0.002273悪化）がそのまま当てはまるとは限らない）

## 設計方針

- 土台は `54_` R0_memofix_plus_LM（444列、Public 0.515030、単層CatBoost現最良）と同一パイプライン
  （`59_`のベースをそのまま使う。ただし`59_`のSLブロック・Plain/Orderedブレンドはここでは使わない）
- 既存のTF-IDF+SVD列（メモ由来15次元含む）は**そのまま残す**。text_featuresは別種の表現なので
  重複よりも補完效果を期待する（重複していれば増分はゼロに近いはずで、それ自体が結果になる）
- **単一の事前登録済み追加**として検証する（[[validation-asymmetry]]: 探索なしの単一修正は
  的中率が相対的に高い）。複数のtext_features組み合わせを検証スコアで選別することはしない
- 在籍月数の回帰ブレンド・最終Platt較正は**第90節で既に検証済み**
  （回帰ブレンド-0.0003・Platt較正+0.0017悪化、いずれもノイズ以下〜悪化）で、
  ここでは組み込まない


In [4]:
!pip install -q catboost optuna janome

In [5]:
import multiprocessing
print(f"CPUコア数: {multiprocessing.cpu_count()}（今回はCPUで学習するため、GPUランタイムは不要）")

CPUコア数: 8（今回はCPUで学習するため、GPUランタイムは不要）


In [6]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))


Mounted at /content/drive


In [7]:
import datetime
import json
import re
import warnings

import numpy as np
import pandas as pd
import catboost as cb
import optuna
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


In [8]:
SCRIPT_NAME = "69_catboost_native_text"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = PROJECT_ROOT / "data" / "output" / "_checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = CHECKPOINT_DIR / f"{SCRIPT_NAME}_checkpoint.csv"

RESET_CHECKPOINT = True  # Trueにすると既存チェックポイントを削除して最初から再計算する
if RESET_CHECKPOINT and CHECKPOINT_PATH.exists():
    CHECKPOINT_PATH.unlink()
    print("チェックポイントを削除しました（全構成を再計算します）")

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"Checkpoint Path: {CHECKPOINT_PATH}")
if CHECKPOINT_PATH.exists():
    logger.info(f"既存のチェックポイントを発見: {len(pd.read_csv(CHECKPOINT_PATH))}件の結果が記録済み")
else:
    logger.info("チェックポイントは未作成（新規実行）")


[2026-08-16 07:59:13] [INFO] === [69_catboost_native_text] 実験開始 ===


INFO:69_catboost_native_text:=== [69_catboost_native_text] 実験開始 ===


[2026-08-16 07:59:14] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260816


INFO:69_catboost_native_text:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260816


[2026-08-16 07:59:14] [INFO] Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/69_catboost_native_text_checkpoint.csv


INFO:69_catboost_native_text:Checkpoint Path: /content/drive/MyDrive/jaggle_2026/data/output/_checkpoints/69_catboost_native_text_checkpoint.csv


[2026-08-16 07:59:14] [INFO] チェックポイントは未作成（新規実行）


INFO:69_catboost_native_text:チェックポイントは未作成（新規実行）


In [9]:
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

# 69_: 在籍月数回帰の目的変数を作るためだけに使う（0-23ヶ月の特徴量には一切触れない＝リークなし）。
#   60_のCoxブロックと同じデータソース。reference/0816版の「在籍月数」目的変数と同じ考え方。
train_monthly_full = pd.read_csv(INPUT_DIR / "employee_monthly_train_full.csv")
TENURE = train_monthly_full.groupby(ID_COL)["経過月数"].max()   # 生存者は119、離職者は退職前の最終月
logger.info(f"在籍月数(TENURE): 平均={TENURE.mean():.1f} 最小={TENURE.min():.0f} 最大={TENURE.max():.0f}")

[2026-08-16 07:59:18] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:69_catboost_native_text:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-16 07:59:18] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:69_catboost_native_text:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-16 07:59:18] [INFO] 定着率: 0.5647


INFO:69_catboost_native_text:定着率: 0.5647


[2026-08-16 07:59:18] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:69_catboost_native_text:Train IDs: 2761, Test IDs: 2502


[2026-08-16 07:59:20] [INFO] 在籍月数(TENURE): 平均=92.3 最小=11 最大=119


INFO:69_catboost_native_text:在籍月数(TENURE): 平均=92.3 最小=11 最大=119


## 0. 早期退職者の特定

`月末在籍状態 == "退職"` の行を持つ社員が、0-23ヶ月の観測期間中に退職した社員。
Train に129名（全員ラベル0）、Test には0名（[[test-set-is-survivor-filtered]]）。


In [10]:
EARLY_LEAVER_IDS = set(train_monthly.loc[train_monthly["月末在籍状態"] == "退職", ID_COL].unique())
_test_early = set(test_monthly.loc[test_monthly["月末在籍状態"] == "退職", ID_COL].unique())

logger.info(f"Train 早期退職者: {len(EARLY_LEAVER_IDS)}名 / {len(train_ids)}名 ({len(EARLY_LEAVER_IDS)/len(train_ids):.1%})")
logger.info(f"Test  早期退職者: {len(_test_early)}名 / {len(test_ids)}名")
_y_idx = train_persona.set_index(ID_COL)[TARGET_COL]
logger.info(f"早期退職者のラベル平均: {_y_idx.loc[list(EARLY_LEAVER_IDS)].mean():.4f}（0.0のはず）")
logger.info(f"定着率: 全体 {y_train.mean():.4f} / 早期退職者を除く {_y_idx[~_y_idx.index.isin(EARLY_LEAVER_IDS)].mean():.4f}")
assert len(_test_early) == 0, "Testに早期退職者が存在する。前提が崩れているので調査すること"


[2026-08-16 07:59:20] [INFO] Train 早期退職者: 129名 / 2761名 (4.7%)


INFO:69_catboost_native_text:Train 早期退職者: 129名 / 2761名 (4.7%)


[2026-08-16 07:59:20] [INFO] Test  早期退職者: 0名 / 2502名


INFO:69_catboost_native_text:Test  早期退職者: 0名 / 2502名


[2026-08-16 07:59:20] [INFO] 早期退職者のラベル平均: 0.0000（0.0のはず）


INFO:69_catboost_native_text:早期退職者のラベル平均: 0.0000（0.0のはず）


[2026-08-16 07:59:20] [INFO] 定着率: 全体 0.5647 / 早期退職者を除く 0.5923


INFO:69_catboost_native_text:定着率: 全体 0.5647 / 早期退職者を除く 0.5923


## 1. 基本特徴量関数の定義（split非依存、`54_`と同一ロジック）

In [11]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_〜18_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_〜18_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_〜18_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ split非依存の基本特徴量関数定義完了")


✅ split非依存の基本特徴量関数定義完了


In [12]:
logger.info("-" * 60)
logger.info("split非依存の基本特徴量を生成中...")
logger.info("-" * 60)

train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("split非依存の基本特徴量生成完了")


[2026-08-16 07:59:20] [INFO] ------------------------------------------------------------


INFO:69_catboost_native_text:------------------------------------------------------------


[2026-08-16 07:59:20] [INFO] split非依存の基本特徴量を生成中...


INFO:69_catboost_native_text:split非依存の基本特徴量を生成中...


[2026-08-16 07:59:20] [INFO] ------------------------------------------------------------


INFO:69_catboost_native_text:------------------------------------------------------------


[2026-08-16 08:04:47] [INFO] split非依存の基本特徴量生成完了


INFO:69_catboost_native_text:split非依存の基本特徴量生成完了


## 2. テキストTF-IDF（A_v1、`54_`と同一・継続採用）

In [13]:
def create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=42):
    '''文字n-gram TF-IDF + TruncatedSVDでテキスト特徴量を生成（Trainのみでfit）'''
    train_text = train_persona[col].fillna("").astype(str)
    test_text = test_persona[col].fillna("").astype(str)

    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=max_features, min_df=min_df)
    train_tfidf = vectorizer.fit_transform(train_text)
    test_tfidf = vectorizer.transform(test_text)

    n_comp = min(n_components, train_tfidf.shape[1] - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=seed, algorithm="arpack")
    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    col_names = [f"{col}_tfidf_svd_{i}" for i in range(n_comp)]
    train_out = pd.DataFrame(train_svd, columns=col_names)
    train_out[ID_COL] = train_persona[ID_COL].values
    test_out = pd.DataFrame(test_svd, columns=col_names)
    test_out[ID_COL] = test_persona[ID_COL].values
    return train_out, test_out, svd.explained_variance_ratio_.sum()

TEXT_COLS = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]

logger.info("テキストTF-IDF+SVD特徴量(A_v1)を生成中...")
tfidf_train_list, tfidf_test_list = [], []
for col in TEXT_COLS:
    tr, te, explained_var = create_tfidf_svd_features(train_persona, test_persona, col, max_features=300, n_components=15, min_df=3, seed=SEED)
    logger.info(f"{col}: SVD累積寄与率={explained_var:.3f}")
    tfidf_train_list.append(tr)
    tfidf_test_list.append(te)

logger.info("テキストTF-IDF+SVD特徴量生成完了")


[2026-08-16 08:04:47] [INFO] テキストTF-IDF+SVD特徴量(A_v1)を生成中...


INFO:69_catboost_native_text:テキストTF-IDF+SVD特徴量(A_v1)を生成中...


[2026-08-16 08:04:48] [INFO] 入社時メモ: SVD累積寄与率=0.760


INFO:69_catboost_native_text:入社時メモ: SVD累積寄与率=0.760


[2026-08-16 08:04:52] [INFO] 上司からのフィードバック: SVD累積寄与率=0.360


INFO:69_catboost_native_text:上司からのフィードバック: SVD累積寄与率=0.360


[2026-08-16 08:04:53] [INFO] 同僚からのフィードバック: SVD累積寄与率=0.421


INFO:69_catboost_native_text:同僚からのフィードバック: SVD累積寄与率=0.421


[2026-08-16 08:04:53] [INFO] テキストTF-IDF+SVD特徴量生成完了


INFO:69_catboost_native_text:テキストTF-IDF+SVD特徴量生成完了


## 3. 四半期/加速度特徴量（D_expanded、`54_`と同一・継続採用）

In [14]:
def create_quarterly_features(monthly_df, employee_ids, metrics, suffix=""):
    quarters = {"q1": (0, 5), "q2": (6, 11), "q3": (12, 17), "q4": (18, 23)}
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数")
        features = {"社員ID": employee_id}
        for metric in metrics:
            q_means = {}
            for qname, (lo, hi) in quarters.items():
                vals = emp_data[emp_data["経過月数"].between(lo, hi)][metric]
                q_means[qname] = vals.mean()
                features[f"{metric}_{qname}_mean{suffix}"] = q_means[qname]
            first_half_delta = q_means["q2"] - q_means["q1"] if pd.notna(q_means["q1"]) and pd.notna(q_means["q2"]) else np.nan
            second_half_delta = q_means["q4"] - q_means["q3"] if pd.notna(q_means["q3"]) and pd.notna(q_means["q4"]) else np.nan
            features[f"{metric}_acceleration{suffix}"] = (
                second_half_delta - first_half_delta if pd.notna(first_half_delta) and pd.notna(second_half_delta) else np.nan
            )
        features_list.append(features)
    return pd.DataFrame(features_list)

D_EXPANDED_METRICS = [
    "残業時間", "有給取得日数", "欠勤日数", "研修時間",
    "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
    "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
    "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
    "顧客満足度評価", "担当プロジェクト数", "月例給与_円",
]

logger.info("四半期/加速度特徴量(D_expanded: 16指標)を生成中...")
train_quarterly_exp = create_quarterly_features(train_monthly, train_ids, D_EXPANDED_METRICS, suffix="_exp")
test_quarterly_exp = create_quarterly_features(test_monthly, test_ids, D_EXPANDED_METRICS, suffix="_exp")
logger.info(f"D_expanded: Train {train_quarterly_exp.shape}, Test {test_quarterly_exp.shape}")


[2026-08-16 08:04:54] [INFO] 四半期/加速度特徴量(D_expanded: 16指標)を生成中...


INFO:69_catboost_native_text:四半期/加速度特徴量(D_expanded: 16指標)を生成中...


[2026-08-16 08:06:49] [INFO] D_expanded: Train (2761, 81), Test (2502, 81)


INFO:69_catboost_native_text:D_expanded: Train (2761, 81), Test (2502, 81)


## 4. Persona単位の基本特徴量（`54_`と同一）

In [15]:
logger.info("Persona単位の基本特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])

for col in TEXT_COLS:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)
train_persona["text_total_chars"] = train_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[TEXT_COLS].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)

train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("Persona単位の基本特徴量処理完了")


[2026-08-16 08:06:50] [INFO] Persona単位の基本特徴量を生成中...


INFO:69_catboost_native_text:Persona単位の基本特徴量を生成中...


[2026-08-16 08:06:50] [INFO] Persona単位の基本特徴量処理完了


INFO:69_catboost_native_text:Persona単位の基本特徴量処理完了


## 5. 転居×勤務地マッチの交互作用特徴量（ブロックL、`49_`でパーサーを修正・`54_`と同一）

In [16]:
def extract_workstyle_section(text):
    if pd.isna(text):
        return None
    m = re.search(r"勤務地・働き方：(.+?)$", text, re.S)
    if m:
        return m.group(1).strip()
    # 49_: 見出しがない書式B（276件、5.24%）のフォールバック。
    lines = [l for l in text.strip().splitlines() if re.search(r"勤務地|転居|在宅勤務", l)]
    return "".join(lines) if lines else None


NEG_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はも]?(許容せず|許容していない|許容しておらず|希望せず|希望しておらず|希望していない)")
POS_RELOC = re.compile(r"転居を伴う(異動|勤務地変更)[はもを]?(許容し?ており|許容)")


def classify_reloc(s):
    if s is None:
        return None
    if NEG_RELOC.search(s):
        return False
    if POS_RELOC.search(s):
        return True
    return None


def extract_desired_location_v1(s):
    '''27_・25_・EDA v3/v4/v5と同一（Public 0.529672で確認済み、カバー率88.6%/train）'''
    if s is None:
        return None
    m = re.search(r"(?:勤務地は|希望勤務地は)(.+?)(?:を希望|。)", s)
    if m:
        return m.group(1)
    m2 = re.search(r"(.+?)を希望勤務地", s)
    return m2.group(1) if m2 else None


def extract_desired_location_v2(s):
    '''v1に「◯◯(勤務|での勤務)?を希望。」パターンを追加した拡張版（カバー率94.1%/train）'''
    if s is None:
        return None
    loc = extract_desired_location_v1(s)
    if loc is None:
        m3 = re.search(r"^([一-龥ぁ-んァ-ンー]+?)(?:での勤務|勤務)?を希望。", s)
        loc = m3.group(1) if m3 else None
    if loc is not None:
        loc = loc.strip("「」")
    return loc


def create_relocation_mismatch_features(persona_df, extract_fn, state_col, flag_col):
    ws_section = persona_df["入社時メモ"].apply(extract_workstyle_section)
    reloc_ok_raw = ws_section.apply(classify_reloc)
    desired = ws_section.apply(extract_fn)
    actual = persona_df["初期勤務地"]
    match = (desired == actual) & desired.notna()

    reloc_true = reloc_ok_raw == True
    reloc_false = reloc_ok_raw == False
    valid = desired.notna() & reloc_ok_raw.notna()

    state = pd.Series("unknown", index=persona_df.index)
    state[valid & reloc_true & match] = "許容_一致"
    state[valid & reloc_true & ~match] = "許容_不一致"
    state[valid & reloc_false & match] = "非許容_一致"
    state[valid & reloc_false & ~match] = "非許容_不一致"

    double_bad = (valid & reloc_false & ~match).astype(int)

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        state_col: state.values,
        flag_col: double_bad.values,
    })

logger.info("転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...")
train_reloc_v1 = create_relocation_mismatch_features(train_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
test_reloc_v1 = create_relocation_mismatch_features(test_persona, extract_desired_location_v1, "転居x勤務地_状態_v1", "転居x勤務地_ダブル悪条件_v1")
train_reloc_v2 = create_relocation_mismatch_features(train_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")
test_reloc_v2 = create_relocation_mismatch_features(test_persona, extract_desired_location_v2, "転居x勤務地_状態_v2", "転居x勤務地_ダブル悪条件_v2")

logger.info(f"L_v1: Train {train_reloc_v1.shape}, Test {test_reloc_v1.shape}")
logger.info(f"L_v2: Train {train_reloc_v2.shape}, Test {test_reloc_v2.shape}")


[2026-08-16 08:06:50] [INFO] 転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


INFO:69_catboost_native_text:転居×勤務地マッチ交互作用特徴量(ブロックL v1/v2)を生成中...


[2026-08-16 08:06:50] [INFO] L_v1: Train (2761, 3), Test (2502, 3)


INFO:69_catboost_native_text:L_v1: Train (2761, 3), Test (2502, 3)


[2026-08-16 08:06:50] [INFO] L_v2: Train (2761, 3), Test (2502, 3)


INFO:69_catboost_native_text:L_v2: Train (2761, 3), Test (2502, 3)


## 6. L2×Mリスク要因数（`54_`で確認済み、Public -0.004119・そのまま採用）

In [17]:
_ANALYTICAL_MAJOR = {"情報", "理工学"}
_ANALYTICAL_JOB = {"IT・エンジニアリング", "データ・商品企画・コンサルティング"}


def create_l2_m_interaction_features(persona_df, reloc_v2_df):
    is_analytical_major = persona_df["専攻分野"].isin(_ANALYTICAL_MAJOR)
    is_analytical_job = persona_df["初期職種"].isin(_ANALYTICAL_JOB)
    m_bad = (~is_analytical_major & is_analytical_job).astype(int)

    state = reloc_v2_df.set_index("社員ID").loc[persona_df["社員ID"], "転居x勤務地_状態_v2"].values
    l2_bad = (state == "非許容_不一致").astype(int)

    both_bad = (l2_bad & m_bad)
    risk_count = l2_bad + m_bad

    return pd.DataFrame({
        "社員ID": persona_df["社員ID"].values,
        "M_不適合": m_bad,
        "L2xM_ダブル不適合": both_bad,
        "L2xM_リスク要因数": risk_count,
    })


train_l2m = create_l2_m_interaction_features(train_persona, train_reloc_v2)
test_l2m = create_l2_m_interaction_features(test_persona, test_reloc_v2)
logger.info(f"L2xMリスク特徴量: Train {train_l2m.shape}, Test {test_l2m.shape}")
print(train_l2m["L2xM_リスク要因数"].value_counts().sort_index())


[2026-08-16 08:06:50] [INFO] L2xMリスク特徴量: Train (2761, 4), Test (2502, 4)


INFO:69_catboost_native_text:L2xMリスク特徴量: Train (2761, 4), Test (2502, 4)


L2xM_リスク要因数
0    2025
1     689
2      47
Name: count, dtype: int64


## 7. 自己学習（詳細）パース特徴量（SLブロック、`59_`で新規追加）

`reference/analysis_0812.ipynb` の `make_learning_features()` を移植。`自己学習（詳細）` 列は
「コース名：X時間｜コース名：Y時間...」形式のテキストで、[[eda-v6-findings]]で「有望・未使用」と
メモしていたが実装していなかった。今回、コースイベント数・ユニークコース数・総学習時間に加え、
コース名のキーワードでDX_IT/対人営業/管理リスク/組織業務の4トピックに分類した時間合計を追加する。

leak安全性: `train_monthly`/`test_monthly` は既に0-23ヶ月に絞り込まれているため、追加のフィルタは不要
（`54_`の他ブロックと同じ前提）。


In [18]:
def create_self_study_features(monthly_df, employee_ids):
    """自己学習（詳細）列を｜区切りでパースし、コースイベント数・総時間・トピック別時間を集計する
    （reference/analysis_0812.ipynb の make_learning_features を移植）。
    """
    z = monthly_df[monthly_df[ID_COL].isin(employee_ids)][[ID_COL, "経過月数", "自己学習（詳細）"]].copy()
    z = z[z["自己学習（詳細）"].notna() & z["自己学習（詳細）"].ne("受講なし")]

    out = pd.DataFrame(index=pd.Index(employee_ids, name=ID_COL))
    out["学習_active_months"] = z.groupby(ID_COL)["経過月数"].nunique()

    tokens = z.assign(token=z["自己学習（詳細）"].astype(str).str.split("｜")).explode("token")
    extracted = tokens["token"].str.extract(r"^(.*)：([0-9.]+)時間$")
    tokens["course"] = extracted[0]
    tokens["hours"] = pd.to_numeric(extracted[1], errors="coerce")
    tokens = tokens.dropna(subset=["course", "hours"])

    out["学習_course_events"] = tokens.groupby(ID_COL).size()
    out["学習_unique_courses"] = tokens.groupby(ID_COL)["course"].nunique()
    out["学習_total_hours_parsed"] = tokens.groupby(ID_COL)["hours"].sum()

    course_buckets = {
        "DX_IT": ["Python", "SQL", "クラウド", "データ", "システム", "RPA", "AI", "統計", "アジャイル"],
        "対人営業": ["顧客", "提案", "交渉", "CRM", "アカウント", "ファシリ", "コミュニケーション"],
        "管理リスク": ["リスク", "コンプライアンス", "内部統制", "労務", "管理会計", "品質管理"],
        "組織業務": ["組織", "人材", "業務", "プロセス", "プロジェクト"],
    }
    course_text = tokens["course"].astype(str)
    for bucket_name, words in course_buckets.items():
        mask = course_text.apply(lambda s: any(word in s for word in words))
        out[f"学習時間_{bucket_name}"] = tokens.loc[mask].groupby(ID_COL)["hours"].sum()

    return out.fillna(0.0).reset_index()


logger.info("自己学習（詳細）パース特徴量(SL)を生成中...")
train_sl = create_self_study_features(train_monthly, train_ids)
test_sl = create_self_study_features(test_monthly, test_ids)
logger.info(f"SL: Train {train_sl.shape}, Test {test_sl.shape}")
print(train_sl.describe().round(2))


[2026-08-16 08:06:50] [INFO] 自己学習（詳細）パース特徴量(SL)を生成中...


INFO:69_catboost_native_text:自己学習（詳細）パース特徴量(SL)を生成中...


[2026-08-16 08:06:51] [INFO] SL: Train (2761, 9), Test (2502, 9)


INFO:69_catboost_native_text:SL: Train (2761, 9), Test (2502, 9)


       学習_active_months  学習_course_events  学習_unique_courses  \
count           2761.00           2761.00            2761.00   
mean               9.34             14.65               9.07   
std                3.33              7.39               4.75   
min                0.00              0.00               0.00   
25%                7.00              9.00               5.00   
50%                9.00             13.00               8.00   
75%               12.00             19.00              12.00   
max               22.00             40.00              25.00   

       学習_total_hours_parsed  学習時間_DX_IT  学習時間_対人営業  学習時間_管理リスク  学習時間_組織業務  
count                2761.00     2761.00    2761.00     2761.00    2761.00  
mean                   40.92       13.11       6.80        8.22       4.60  
std                    17.97       11.69      10.17        8.35       5.83  
min                     0.00        0.00       0.00        0.00       0.00  
25%                    28.00        4.

## 7b. メモtxt / 学習txt（`69_`で新規追加、text_features用）

`reference/長期定着予測_crmaine_0816.ipynb`の設計を移植。CatBoostのtext_features機構に
渡すため、TF-IDF+SVDのような固定次元圧縮はせず、生テキストを軽く前処理しただけの
文字列を新しい列として追加する。


In [19]:
from janome.tokenizer import Tokenizer
import re as _re

_tokenizer = Tokenizer()
_KEEP_POS = {"名詞", "動詞", "形容詞"}


def tokenize_memo(text):
    """入社時メモを形態素解析し、名詞・動詞・形容詞の原形だけを空白区切りで連結する。
    （reference/長期定着予測_crmaine_0816.ipynb の「文書分割」を移植）
    """
    if pd.isna(text):
        return ""
    return " ".join(w.base_form for w in _tokenizer.tokenize(str(text))
                     if w.part_of_speech.split(",")[0] in _KEEP_POS)


def create_memo_text(persona_df):
    return pd.DataFrame({
        ID_COL: persona_df[ID_COL].values,
        "メモtxt": persona_df["入社時メモ"].apply(tokenize_memo).values,
    })


def extract_course_themes(text):
    """「管理会計基礎：12.5時間｜統計学基礎：7.0時間」→ ["管理会計基礎","統計学基礎"]"""
    if pd.isna(text) or text == "受講なし":
        return []
    return [n.strip().replace(" ", "") for n in _re.findall(r"([^｜：]+)：[\d.]+時間", str(text))]


def create_study_theme_text(monthly_df, employee_ids):
    """24ヶ月分の自己学習テーマ名を全て集めて空白区切りで連結する（社員1名=1行）。"""
    m = monthly_df[monthly_df[ID_COL].isin(employee_ids)][[ID_COL, "自己学習（詳細）"]].copy()
    m["テーマ一覧"] = m["自己学習（詳細）"].apply(extract_course_themes)
    joined = (m.groupby(ID_COL)["テーマ一覧"]
              .apply(lambda lists: " ".join(t for lst in lists for t in lst)))
    joined = joined.reindex(employee_ids).fillna("")
    return pd.DataFrame({ID_COL: employee_ids, "学習txt": joined.values})


logger.info("メモtxt・学習txt(text_features用)を生成中...")
train_memotxt = create_memo_text(train_persona)
test_memotxt = create_memo_text(test_persona)
train_studytxt = create_study_theme_text(train_monthly, train_ids)
test_studytxt = create_study_theme_text(test_monthly, test_ids)
logger.info(f"メモtxt: Train {train_memotxt.shape}, Test {test_memotxt.shape}")
logger.info(f"学習txt: Train {train_studytxt.shape}, Test {test_studytxt.shape}")
print("メモtxt サンプル:", train_memotxt["メモtxt"].iloc[0][:80])
print("学習txt サンプル:", train_studytxt.loc[train_studytxt["学習txt"] != "", "学習txt"].iloc[0][:80])


[2026-08-16 08:06:52] [INFO] メモtxt・学習txt(text_features用)を生成中...


INFO:69_catboost_native_text:メモtxt・学習txt(text_features用)を生成中...


[2026-08-16 08:07:20] [INFO] メモtxt: Train (2761, 2), Test (2502, 2)


INFO:69_catboost_native_text:メモtxt: Train (2761, 2), Test (2502, 2)


[2026-08-16 08:07:20] [INFO] 学習txt: Train (2761, 2), Test (2502, 2)


INFO:69_catboost_native_text:学習txt: Train (2761, 2), Test (2502, 2)


メモtxt サンプル: 経歴 大学 卒 人文 教養 中途 入社 前 職 初期 職種 いずれ コーポレート 人物 所見 必要 情報 周囲 開く 認識 丁寧 合わせる 協 働 進める 姿勢
学習txt サンプル: 管理会計基礎 統計学基礎 組織開発・人材育成 統計学基礎 統計学基礎 労務管理基礎 労務管理基礎 業務改善の進め方 労務管理基礎 情報セキュリティ基礎 労務管理


## 8. 部署Target Encoding（リーク対策済）と `prepare_split` 関数

`54_`と同一ロジックに、`extra_blocks`へ`"SL"`を追加した場合の分岐だけを足す。


In [20]:
def create_department_target_encoding(train_persona, test_persona, y_train, fit_ids, seed=42, n_splits=5, smoothing=10):
    '''初期部署IDのKFold + スムージング付きTarget Encoding（15_〜18_の修正版と同一ロジック）'''
    col = "初期部署ID"
    is_fit = train_persona[ID_COL].isin(fit_ids).values
    dept_all = train_persona[col].values
    y_arr = y_train.values
    global_mean = y_arr[is_fit].mean()

    fit_indices = np.where(is_fit)[0]
    dept_fit = dept_all[fit_indices]
    y_fit = y_arr[fit_indices]

    train_te = np.full(len(train_persona), global_mean)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, val_idx in kf.split(fit_indices):
        df_tr = pd.DataFrame({col: dept_fit[tr_idx], "y": y_fit[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        actual_val_idx = fit_indices[val_idx]
        train_te[actual_val_idx] = pd.Series(dept_fit[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_fit, "y": y_fit})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    not_fit_indices = np.where(~is_fit)[0]
    train_te[not_fit_indices] = pd.Series(dept_all[not_fit_indices]).map(mapping_full).fillna(global_mean).values

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        ID_COL: train_persona[ID_COL].values,
        "dept_target_enc": train_te,
        "dept_size": pd.Series(dept_all).map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        ID_COL: test_persona[ID_COL].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def prepare_split(split_ratio, extra_blocks=None, exclude_early_from_val=True):
    '''指定した分割比率で特徴量を組み立てる（54_と同一 + SLブロック対応）。'''
    extra_blocks = extra_blocks or set()
    sorted_persona = train_persona.sort_values("入社日")
    split_point = int(len(sorted_persona) * split_ratio)
    train_period_ids = set(sorted_persona.iloc[:split_point][ID_COL])

    train_dept_te, test_dept_te = create_department_target_encoding(
        train_persona, test_persona, y_train, fit_ids=train_period_ids, seed=SEED, n_splits=5, smoothing=10
    )

    train_persona_features = train_persona.drop(columns=[TARGET_COL])
    tf = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
    tf = tf.merge(train_cat_change, on=ID_COL, how="left")
    tf = tf.merge(train_missing, on=ID_COL, how="left")
    tf = tf.merge(train_domain, on=ID_COL, how="left")
    tf = tf.merge(train_advanced_stats, on=ID_COL, how="left")
    tf = tf.merge(train_cluster, on=ID_COL, how="left")
    tf = tf.merge(train_dept_te, on=ID_COL, how="left")
    tf = tf.merge(train_eda_feats, on=ID_COL, how="left")
    tf = tf.merge(train_mgr, on=ID_COL, how="left")
    tf = tf.merge(train_quarterly_exp, on=ID_COL, how="left")
    for trdf in tfidf_train_list:
        tf = tf.merge(trdf, on=ID_COL, how="left")

    ttf = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
    ttf = ttf.merge(test_cat_change, on=ID_COL, how="left")
    ttf = ttf.merge(test_missing, on=ID_COL, how="left")
    ttf = ttf.merge(test_domain, on=ID_COL, how="left")
    ttf = ttf.merge(test_advanced_stats, on=ID_COL, how="left")
    ttf = ttf.merge(test_cluster, on=ID_COL, how="left")
    ttf = ttf.merge(test_dept_te, on=ID_COL, how="left")
    ttf = ttf.merge(test_eda_feats, on=ID_COL, how="left")
    ttf = ttf.merge(test_mgr, on=ID_COL, how="left")
    ttf = ttf.merge(test_quarterly_exp, on=ID_COL, how="left")
    for tedf in tfidf_test_list:
        ttf = ttf.merge(tedf, on=ID_COL, how="left")

    if "L2" in extra_blocks:
        tf = tf.merge(train_reloc_v2, on=ID_COL, how="left")
        tf = tf.merge(train_l2m, on=ID_COL, how="left")
        ttf = ttf.merge(test_reloc_v2, on=ID_COL, how="left")
        ttf = ttf.merge(test_l2m, on=ID_COL, how="left")

    if "SL" in extra_blocks:
        tf = tf.merge(train_sl, on=ID_COL, how="left")
        ttf = ttf.merge(test_sl, on=ID_COL, how="left")

    if "TXT" in extra_blocks:
        tf = tf.merge(train_memotxt, on=ID_COL, how="left")
        tf = tf.merge(train_studytxt, on=ID_COL, how="left")
        ttf = ttf.merge(test_memotxt, on=ID_COL, how="left")
        ttf = ttf.merge(test_studytxt, on=ID_COL, how="left")

    _train_period_features = tf[tf[ID_COL].isin(train_period_ids)]
    job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
    job_means = {m: _train_period_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
    category_means_train = {m: _train_period_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}
    grade_salary_mean = _train_period_features.groupby("初期等級")["初任給_円"].mean().to_dict()
    category_salary_mean = _train_period_features.groupby("入社区分")["初任給_円"].mean().to_dict()
    grade_monthly_salary_mean = _train_period_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

    for df_ in [tf, ttf]:
        for m in job_dev_metrics:
            df_[f"{m}_job_deviation"] = df_[m] - df_["初期職種"].map(job_means[m])
        df_["研修時間_職種比"] = df_["研修時間_mean"] / df_["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
        df_["研修時間_区分比"] = df_["研修時間_mean"] / df_["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)
        df_["初任給_等級内偏差"] = df_["初任給_円"] - df_["初期等級"].map(grade_salary_mean)
        df_["初任給_区分内偏差"] = df_["初任給_円"] - df_["入社区分"].map(category_salary_mean)
        df_["月例給与_等級内偏差"] = df_["月例給与_円_mean"] - df_["初期等級"].map(grade_monthly_salary_mean)

    drop_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
                 "初期部署ID", "初期等級", "最終学歴", "前職職種"]
    tf = tf.drop(columns=[c for c in drop_cols if c in tf.columns]).set_index(ID_COL)
    ttf = ttf.drop(columns=[c for c in drop_cols if c in ttf.columns]).set_index(ID_COL)

    target_series = train_persona.set_index(ID_COL)[TARGET_COL]
    tf_sorted = tf.sort_values("入社日")
    y_sorted = target_series.loc[tf_sorted.index]

    ag_train = tf_sorted.iloc[:split_point].copy()
    ag_tuning = tf_sorted.iloc[split_point:].copy()
    ag_train[TARGET_COL] = y_sorted.iloc[:split_point].values
    ag_tuning[TARGET_COL] = y_sorted.iloc[split_point:].values

    if exclude_early_from_val and len(ag_tuning) > 0:
        n_before = len(ag_tuning)
        ag_tuning = ag_tuning[~ag_tuning.index.isin(EARLY_LEAVER_IDS)]
        logger.info(f"  検証セット: {n_before} → {len(ag_tuning)}件（早期退職者{n_before - len(ag_tuning)}名を除外）")

    return ag_train, ag_tuning, ttf

print("✅ 部署Target Encoding・prepare_split関数定義完了（SLブロック対応版）")


✅ 部署Target Encoding・prepare_split関数定義完了（SLブロック対応版）


## 9. 特徴量の組み立て（extra_blocks={"L2","SL"}を常時マージし、CONFIGSで列選択を切替）

In [21]:
def _feature_cols(df):
    return [c for c in df.columns if c not in ["入社日", TARGET_COL]]


BLOCK = {"L2", "SL", "TXT"}

logger.info("=" * 60)
logger.info("[検証用] split_80_20 / 検証=生存者のみ")
ag_train_80b, ag_val_surv, _ = prepare_split(0.8, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("[提出用] 全件学習（検証セットなし）")
ag_full, ag_empty, test_features_full = prepare_split(1.0, extra_blocks=BLOCK, exclude_early_from_val=True)

logger.info("-" * 60)
logger.info(f"main_train={len(ag_train_80b)}, main_valid(生存者)={len(ag_val_surv)}")
logger.info(f"全件={len(ag_full)}")
logger.info(f"特徴量数: {len(_feature_cols(ag_train_80b))}")
assert len(ag_empty) == 0, "全件学習のときは検証セットが空のはず"


[2026-08-16 08:07:20] [INFO] ============================================================


INFO:69_catboost_native_text:============================================================


[2026-08-16 08:07:20] [INFO] [検証用] split_80_20 / 検証=生存者のみ


INFO:69_catboost_native_text:[検証用] split_80_20 / 検証=生存者のみ


[2026-08-16 08:07:21] [INFO]   検証セット: 553 → 535件（早期退職者18名を除外）


...

INFO:69_catboost_native_text:main_train=2208, main_valid(生存者)=535


[2026-08-16 08:07:21] [INFO] 全件=2761


INFO:69_catboost_native_text:全件=2761


[2026-08-16 08:07:21] [INFO] 特徴量数: 454


INFO:69_catboost_native_text:特徴量数: 454


## 10. 特徴量グループの棚卸し（SLグループを追加）

In [22]:
ALL_FEATS = set(_feature_cols(ag_train_80b))

DERIVED_COLS = [
    "残業時間_mean_job_deviation", "研修時間_mean_job_deviation", "360度評価_親和度_mean_job_deviation",
    "研修時間_職種比", "研修時間_区分比",
    "初任給_等級内偏差", "初任給_区分内偏差", "月例給与_等級内偏差",
]
DEPT_TE_COLS = ["dept_target_enc", "dept_size"]


def _cols_of(df):
    return [c for c in df.columns if c != ID_COL]


_RAW_GROUPS = {
    "persona":   [c for c in train_persona.columns if c not in (ID_COL, TARGET_COL)],
    "agg":       _cols_of(train_monthly_agg),
    "catchange": _cols_of(train_cat_change),
    "missing":   _cols_of(train_missing),
    "domain":    _cols_of(train_domain),
    "advstats":  _cols_of(train_advanced_stats),
    "cluster":   _cols_of(train_cluster),
    "deptte":    DEPT_TE_COLS,
    "edafeat":   _cols_of(train_eda_feats),
    "mgr":       _cols_of(train_mgr),
    "quarterly": _cols_of(train_quarterly_exp),
    "tfidf":     [c for _df in tfidf_train_list for c in _cols_of(_df)],
    "L2":        _cols_of(train_reloc_v2),
    "LM":        _cols_of(train_l2m),
    "SL":        _cols_of(train_sl),
    "TXT":       ["メモtxt", "学習txt"],
    "derived":   DERIVED_COLS,
}

FEATURE_GROUPS = {g: [c for c in cols if c in ALL_FEATS] for g, cols in _RAW_GROUPS.items()}
ALL_GROUPS = set(FEATURE_GROUPS)

_covered = [c for cols in FEATURE_GROUPS.values() for c in cols]
_dupes = sorted({c for c in _covered if _covered.count(c) > 1})
assert not _dupes, f"複数グループに重複している列: {_dupes}"
_orphans = sorted(ALL_FEATS - set(_covered))
assert not _orphans, f"どのグループにも属さない列: {_orphans}"

print(f"特徴量 合計 {len(ALL_FEATS)} 列")
print("-" * 52)
for g in sorted(FEATURE_GROUPS, key=lambda x: -len(FEATURE_GROUPS[x])):
    print(f"  {g:<10s} {len(FEATURE_GROUPS[g]):>4d} 列   例: {FEATURE_GROUPS[g][:2]}")
print("-" * 52)
print("✅ グループ分類は全列を過不足なく覆っている")


特徴量 合計 454 列
----------------------------------------------------
  agg         224 列   例: ['残業時間_mean', '残業時間_std']
  quarterly    80 列   例: ['残業時間_q1_mean_exp', '残業時間_q2_mean_exp']
  tfidf        45 列   例: ['入社時メモ_tfidf_svd_0', '入社時メモ_tfidf_svd_1']
  advstats     25 列   例: ['残業時間_skew', '残業時間_kurtosis']
  persona      21 列   例: ['入社区分', '入社時年齢']
  catchange    14 列   例: ['部署ID_changes', '部署ID_unique_count']
  edafeat      11 列   例: ['欠勤発生月数', '欠勤_最長連続月数']
  SL            8 列   例: ['学習_active_months', '学習_course_events']
  derived       8 列   例: ['残業時間_mean_job_deviation', '研修時間_mean_job_deviation']
  missing       4 列   例: ['360度評価_親和度_missing_rate', '360度評価_信頼度_missing_rate']
  domain        3 列   例: ['engagement_score', 'overtime_stability']
  LM            3 列   例: ['M_不適合', 'L2xM_ダブル不適合']
  deptte        2 列   例: ['dept_target_enc', 'dept_size']
  L2            2 列   例: ['転居x勤務地_状態_v2', '転居x勤務地_ダブル悪条件_v2']
  TXT           2 列   例: ['メモtxt', '学習txt']
  cluster       1 列   例: ['clu

## 11. 構成の事前登録

`54_`のA_PARAMS/反復数をそのまま流用する（ハイパラ探索はしない＝「text_featuresだけの差」にするため）。


In [23]:
A_PARAMS = {
    "depth": 4,
    "learning_rate": 0.03518359458951149,
    "l2_leaf_reg": 2.217690447016724,
    "border_count": 218,
    "bagging_temperature": 0.6787467566574921,
    "random_strength": 1.438494697238285,
}

ITER_HOLDOUT = 560   # 38_・54_と同一
ITER_FULL    = 560

SEEDS_SUB = [42, 2024, 7, 1234, 99]
SEEDS_VAL = [42, 2024, 7, 1234, 99, 555, 31337, 2718]

CONFIGS = {
    "R0_memofix_plus_LM":       {"groups": ALL_GROUPS - {"SL", "TXT"}},   # 54_のベスト構成の再現（444列）
    "R0_plus_LM_plus_TXT":      {"groups": ALL_GROUPS - {"SL"}},          # + メモtxt/学習txt (text_features)
}

VAL_REJECT_MARGIN = 0.02


def cols_for(spec, df):
    keep = set()
    for g in spec["groups"]:
        keep |= set(FEATURE_GROUPS[g])
    return [c for c in _feature_cols(df) if c in keep]


print(f"{'config':<28s} {'列数':>5s}")
print("-" * 40)
for name, spec in CONFIGS.items():
    print(f"{name:<28s} {len(cols_for(spec, ag_train_80b)):>5d}")


config                          列数
----------------------------------------
R0_memofix_plus_LM             444
R0_plus_LM_plus_TXT            446


## 12. モデル関数（`54_`ベース + text_features対応）

`text_feats`に含まれる列は`cat_features`から除外し、`CatBoostClassifier`の`text_features`引数に渡す。
NaNの埋め方も分ける: 数値・カテゴリ列は`-999`（既存踏襲）、テキスト列は空文字列（text_featuresは
非nullの文字列を要求するため）。`text_feats=[]`のときは既存動作と完全に一致する
（`R0_memofix_plus_LM`のval=0.505477が再現されることで確認する）。


In [24]:
def _fit_one(X_tr, y_tr, obj_cols, txt_cols, params, n_iter, seed):
    model = cb.CatBoostClassifier(
        **params, iterations=int(n_iter), random_seed=seed,
        verbose=False, cat_features=obj_cols, text_features=txt_cols, task_type="CPU",
    )
    model.fit(X_tr, y_tr)
    return model


def _prep_X(df, feature_cols, txt_cols):
    """text_features列は空文字列で、それ以外は-999で欠損を埋める（既存と同じ規約を維持）。"""
    X = df[feature_cols].copy()
    non_txt = [c for c in feature_cols if c not in txt_cols]
    X[non_txt] = X[non_txt].fillna(-999)
    for c in txt_cols:
        X[c] = X[c].fillna("")
    return X


def fit_holdout_fixed(ag_train, ag_val, feature_cols, txt_cols, params, n_iter, seeds):
    """80/20ホールドアウトを反復数固定で学習し、シードごとの検証予測を返す（54_と同一設計）。"""
    obj_cols = [c for c in feature_cols
                if c not in txt_cols and ag_train[c].dtype == "object"]
    X_tr, y_tr = _prep_X(ag_train, feature_cols, txt_cols), ag_train[TARGET_COL]
    X_va, y_va = _prep_X(ag_val, feature_cols, txt_cols), ag_val[TARGET_COL]

    val_preds = []
    for seed in seeds:
        model = _fit_one(X_tr, y_tr, obj_cols, txt_cols, params, n_iter, seed)
        val_preds.append(model.predict_proba(X_va)[:, 1])
    val_preds = np.array(val_preds)

    singles = [log_loss(y_va, vp) for vp in val_preds]
    return {
        "val_seedavg": float(log_loss(y_va, val_preds.mean(axis=0))),
        "val_single_mean": float(np.mean(singles)),
        "val_single_sd": float(np.std(singles)),
        "val_preds": val_preds,
        "y_val": y_va.values,
    }


def fit_full_fixed(ag_full, test_feats, feature_cols, txt_cols, params, n_iter, seeds):
    """Train全件で学習して Test を予測する。"""
    obj_cols = [c for c in feature_cols
                if c not in txt_cols and ag_full[c].dtype == "object"]
    X_tr, y_tr = _prep_X(ag_full, feature_cols, txt_cols), ag_full[TARGET_COL]
    X_test = _prep_X(test_feats, feature_cols, txt_cols)

    test_preds = []
    for seed in seeds:
        model = _fit_one(X_tr, y_tr, obj_cols, txt_cols, params, n_iter, seed)
        test_preds.append(model.predict_proba(X_test)[:, 1])
        logger.info(f"    seed={seed}: 全件学習完了")
    return np.array(test_preds)


def save_submission(test_index, preds, config_label):
    path = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}.csv"
    pd.DataFrame({ID_COL: test_index, TARGET_COL: preds}).to_csv(path, index=False, header=False)
    logger.info(f"  提出ファイル: {path.name}（予測平均={preds.mean():.4f}）")
    return str(path)

print("✅ モデル関数定義完了（text_features対応版）")


✅ モデル関数定義完了（text_features対応版）


## 12b. 在籍月数の回帰・Platt較正・ブレンド（`69_`で新規追加）

`reference/長期定着予測_crmaine_0816.ipynb`（現1位が過去に使用していたと見られる）の設計を移植。
第90節で既にこの技法単体を検証し「回帰ブレンド-0.0003・最終Platt較正+0.0017悪化」という
否定的な結果を得ているが、**text_features（メモtxt/学習txt）と組み合わせると相互作用で
プラスに転じる可能性がある**というユーザーの仮説を検証するため、baseline・+TXTの両方に対して
この技法を追加した「+BLEND」構成を作る（2×2の要因計画）。

### 設計上の注意（reference/0816版を踏襲した箇所）

- 回帰→確率のPlatt較正器は検証セット自身でfit&applyする（in-sample）。1特徴量・2パラメータの
  単純なロジスティック回帰なので過学習リスクは小さいと考えられるが、`blend_val_logloss`の数値には
  わずかな楽観バイアスが乗りうる点は注記する。**最終Platt較正の方はネストKFoldでOOFにしている**
  （`calibrated_val_logloss`が最終的な採否判断に使う指標）
- 自分たちの検証セット(`ag_val_surv`)は既に早期退職者を除外済み（[[test-set-is-survivor-filtered]]）
  なので、reference側にあった「生存者のみでPlattをfitする」というフィルタは不要
  （検証セット全体が既に対象人数と一致する）
- ブレンド比率は reference と同じ **85:15（分類器:回帰）で固定**。ここで走査すると
  Public評価に対する過学習になる（[[private-lb-variance-strategy]]）


In [25]:
def fit_holdout_regressor(ag_train, ag_val, feature_cols, txt_cols, params, n_iter, seeds):
    """在籍月数(TENURE)を目的変数にしたCatBoostRegressor。分類器と全く同じ特徴量・ハイパラ
    （loss_functionだけRMSEに差し替え）。reference/0816版の「回帰設定」と同じ設計。"""
    obj_cols = [c for c in feature_cols if c not in txt_cols and ag_train[c].dtype == "object"]
    X_tr = _prep_X(ag_train, feature_cols, txt_cols)
    y_tr = TENURE.reindex(ag_train.index)
    assert y_tr.notna().all(), "TENUREにマッチしない社員IDがある"
    X_va = _prep_X(ag_val, feature_cols, txt_cols)

    val_preds = []
    for seed in seeds:
        model = cb.CatBoostRegressor(**params, loss_function="RMSE", eval_metric="RMSE",
                                     iterations=int(n_iter), random_seed=seed, verbose=False,
                                     cat_features=obj_cols, text_features=txt_cols, task_type="CPU")
        model.fit(X_tr, y_tr)
        val_preds.append(model.predict(X_va))
    return np.mean(val_preds, axis=0)


def fit_full_regressor(ag_full, test_feats, feature_cols, txt_cols, params, n_iter, seeds):
    obj_cols = [c for c in feature_cols if c not in txt_cols and ag_full[c].dtype == "object"]
    X_tr = _prep_X(ag_full, feature_cols, txt_cols)
    y_tr = TENURE.reindex(ag_full.index)
    assert y_tr.notna().all(), "TENUREにマッチしない社員IDがある"
    X_test = _prep_X(test_feats, feature_cols, txt_cols)

    test_preds = []
    for seed in seeds:
        model = cb.CatBoostRegressor(**params, loss_function="RMSE", eval_metric="RMSE",
                                     iterations=int(n_iter), random_seed=seed, verbose=False,
                                     cat_features=obj_cols, text_features=txt_cols, task_type="CPU")
        model.fit(X_tr, y_tr)
        test_preds.append(model.predict(X_test))
        logger.info(f"    [回帰] seed={seed}: 全件学習完了")
    return np.mean(test_preds, axis=0)


def _to_logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def _platt_fit(x, y):
    lr = LogisticRegression(max_iter=1000)
    lr.fit(np.asarray(x).reshape(-1, 1), y)
    return lr


def _platt_apply(lr, x):
    return lr.predict_proba(np.asarray(x).reshape(-1, 1))[:, 1]


MIX_RATIO = 0.85   # reference/0816版と同じ固定値（分類器85%:回帰15%）。ここでは走査しない


def apply_regression_blend(cls_val, cls_test, reg_val, reg_test, y_val):
    """回帰→確率(Platt)→85:15ブレンド→最終Platt較正、をreference/0816版と同じ手順で適用する。"""
    reg_platt = _platt_fit(reg_val, y_val)
    reg_prob_val = _platt_apply(reg_platt, reg_val)
    reg_prob_test = _platt_apply(reg_platt, reg_test)

    blend_val = MIX_RATIO * cls_val + (1 - MIX_RATIO) * reg_prob_val
    blend_test = MIX_RATIO * cls_test + (1 - MIX_RATIO) * reg_prob_test

    # 最終Platt較正: 検証セット内でネストKFold(5, seed=27)を回し、OOFの較正スコアを得る
    z_val = _to_logit(blend_val)
    kf = KFold(n_splits=5, shuffle=True, random_state=27)
    calibrated_val = np.zeros(len(z_val))
    for tr_idx, va_idx in kf.split(z_val):
        cal = _platt_fit(z_val[tr_idx], y_val[tr_idx])
        calibrated_val[va_idx] = _platt_apply(cal, z_val[va_idx])

    # 提出用の最終較正器は検証セット全体で1本fitする
    final_cal = _platt_fit(z_val, y_val)
    calibrated_test = _platt_apply(final_cal, _to_logit(blend_test))

    return {
        "blend_val": blend_val, "blend_test": blend_test,
        "calibrated_val": calibrated_val, "calibrated_test": calibrated_test,
        "reg_prob_val_logloss": float(log_loss(y_val, reg_prob_val)),
        "blend_val_logloss": float(log_loss(y_val, blend_val)),
        "calibrated_val_logloss": float(log_loss(y_val, calibrated_val)),
    }

print("✅ 在籍月数回帰・Platt較正・ブレンド関数定義完了")


✅ 在籍月数回帰・Platt較正・ブレンド関数定義完了


## 13. チェックポイント

In [26]:
RESULT_SCHEMA = [
    "config", "kind", "n_features", "n_text_features",
    "val_seedavg", "val_single_mean", "val_single_sd",
    "n_iterations", "n_train", "pred_mean", "submission_path",
]


def make_row(**kwargs):
    unknown = set(kwargs) - set(RESULT_SCHEMA)
    assert not unknown, f"RESULT_SCHEMAに無いキー: {unknown}"
    row = {k: np.nan for k in RESULT_SCHEMA}
    row.update(kwargs)
    return row


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        return pd.read_csv(CHECKPOINT_PATH)
    return pd.DataFrame(columns=RESULT_SCHEMA)


def save_checkpoint_row(result):
    df = pd.DataFrame([result])[RESULT_SCHEMA]
    write_header = not CHECKPOINT_PATH.exists()
    df.to_csv(CHECKPOINT_PATH, mode="a", header=write_header, index=False)


def run_or_resume(config_label, run_fn):
    checkpoint = load_checkpoint()
    existing = checkpoint[checkpoint["config"] == config_label] if len(checkpoint) else checkpoint
    if len(existing) > 0:
        row = existing.iloc[0].to_dict()
        logger.info(f"[{config_label}] チェックポイントから復元: val_seedavg={row.get('val_seedavg')}")
        return row
    result = run_fn()
    save_checkpoint_row(result)
    return result

print("✅ チェックポイント関数定義完了")


✅ チェックポイント関数定義完了


## 14. 実行: baseline / +TXT の標準構成（2つ）


In [27]:
TXT_COLS = set(FEATURE_GROUPS["TXT"])


def make_runner(config_label, spec):
    def _run():
        feats = cols_for(spec, ag_train_80b)
        feats_full = cols_for(spec, ag_full)
        assert feats == feats_full, "検証と全件学習で特徴量列が食い違っている"
        txt_cols = [c for c in feats if c in TXT_COLS]

        logger.info("=" * 60)
        logger.info(f"[{config_label}] {len(feats)}列（うちtext_features {len(txt_cols)}列: {txt_cols}）")

        hold = fit_holdout_fixed(ag_train_80b, ag_val_surv, feats, txt_cols, A_PARAMS, ITER_HOLDOUT, SEEDS_VAL)
        logger.info(f"  検証(生存者{len(ag_val_surv)}名): シード平均 {hold['val_seedavg']:.6f} "
                    f"/ 単一シード {hold['val_single_mean']:.6f} ± {hold['val_single_sd']:.6f}")
        np.save(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_valpreds.npy", hold["val_preds"])

        test_preds = fit_full_fixed(ag_full, test_features_full, feats, txt_cols, A_PARAMS, ITER_FULL, SEEDS_SUB)
        preds = test_preds.mean(axis=0)
        path = save_submission(test_features_full.index, preds, config_label)

        return make_row(
            config=config_label, kind="standard", n_features=len(feats), n_text_features=len(txt_cols),
            val_seedavg=hold["val_seedavg"], val_single_mean=hold["val_single_mean"],
            val_single_sd=hold["val_single_sd"],
            n_iterations=ITER_FULL, n_train=len(ag_full), pred_mean=float(preds.mean()),
            submission_path=path,
        )
    return _run


standard_results = {}
for _name, _spec in CONFIGS.items():
    standard_results[_name] = run_or_resume(_name, make_runner(_name, _spec))

print()
print(f"{'config':<28s} {'列数':>5s} {'text列':>6s} {'val(8シード平均)':>16s} {'単一sd':>9s}")
print("-" * 68)
for _name, _r in standard_results.items():
    print(f"{_name:<28s} {int(_r['n_features']):>5d} {int(_r['n_text_features']):>6d} "
          f"{float(_r['val_seedavg']):>16.6f} {float(_r['val_single_sd']):>9.6f}")


[2026-08-16 08:07:22] [INFO] ============================================================


INFO:69_catboost_native_text:============================================================


[2026-08-16 08:07:22] [INFO] [R0_memofix_plus_LM] 444列（うちtext_features 0列: []）


INFO:69_catboost_native_text:[R0_memofix_plus_LM] 444列（うちtext_features 0列: []）


[2026-08-16 08:07:57] [INFO]   検証(生存者535名): シード平均 0.505477 / 単一シード 0.508987 ± 0.005181


...

[2026-08-16 08:10:28] [INFO]     seed=99: 全件学習完了


INFO:69_catboost_native_text:    seed=99: 全件学習完了


[2026-08-16 08:10:28] [INFO]   提出ファイル: 20260816_69_catboost_native_text_R0_plus_LM_plus_TXT.csv（予測平均=0.5899）


INFO:69_catboost_native_text:  提出ファイル: 20260816_69_catboost_native_text_R0_plus_LM_plus_TXT.csv（予測平均=0.5899）



config                          列数  text列      val(8シード平均)      単一sd
--------------------------------------------------------------------
R0_memofix_plus_LM             444      0         0.505477  0.005181
R0_plus_LM_plus_TXT            446      2         0.511183  0.006429


## 14b. 実行: 在籍月数回帰ブレンド+Platt較正を追加した構成（2つ、`69_`で新規追加）

baseline・+TXTのそれぞれに対して、在籍月数の回帰ブレンド+最終Platt較正を追加する
（2×2の要因計画。ユーザーの仮説「text_featuresと組み合わせると相互作用でプラスに転じるか」を
直接検証する）。分類器の予測は直前のセルが保存した`.npy`/提出CSVから読み直す（再学習しない）。


In [28]:
def load_cls_preds(config_label):
    """直前の標準run が保存した valpreds.npy / 提出CSV から、分類器のシード平均予測を読み直す。
    checkpointから復元された場合でも同じロジックで再取得できる（再学習しない）。"""
    val_arr = np.load(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_{config_label}_valpreds.npy").mean(axis=0)
    test_path = standard_results[config_label]["submission_path"]
    test_arr = (pd.read_csv(test_path, header=None, names=[ID_COL, "p"])
                .set_index(ID_COL)["p"].reindex(test_features_full.index).values)
    return val_arr, test_arr


def make_blend_runner(config_label, base_config, spec):
    def _run():
        feats = cols_for(spec, ag_train_80b)
        txt_cols = [c for c in feats if c in TXT_COLS]
        cls_val, cls_test = load_cls_preds(base_config)

        logger.info("=" * 60)
        logger.info(f"[{config_label}] 在籍月数回帰+Platt較正+ブレンドを追加（土台: {base_config}）")

        reg_val = fit_holdout_regressor(ag_train_80b, ag_val_surv, feats, txt_cols, A_PARAMS, ITER_HOLDOUT, SEEDS_VAL)
        reg_test = fit_full_regressor(ag_full, test_features_full, feats, txt_cols, A_PARAMS, ITER_FULL, SEEDS_SUB)

        y_val_arr = ag_val_surv[TARGET_COL].values
        r = apply_regression_blend(cls_val, cls_test, reg_val, reg_test, y_val_arr)
        logger.info(f"  回帰→確率単体          : {r['reg_prob_val_logloss']:.6f}")
        logger.info(f"  ブレンド後({int(MIX_RATIO*100)}:{int((1-MIX_RATIO)*100)})       : {r['blend_val_logloss']:.6f}")
        logger.info(f"  最終Platt較正後(OOF)   : {r['calibrated_val_logloss']:.6f}")

        path = save_submission(test_features_full.index, r["calibrated_test"], config_label)
        return make_row(
            config=config_label, kind="blend", n_features=len(feats), n_text_features=len(txt_cols),
            val_seedavg=r["calibrated_val_logloss"],
            n_iterations=ITER_FULL, n_train=len(ag_full), pred_mean=float(r["calibrated_test"].mean()),
            submission_path=path,
        )
    return _run


BLEND_CONFIGS = {
    "R0_memofix_plus_LM_BLEND":  "R0_memofix_plus_LM",
    "R0_plus_LM_plus_TXT_BLEND": "R0_plus_LM_plus_TXT",
}

blend_results = {}
for _blend_name, _base_name in BLEND_CONFIGS.items():
    blend_results[_blend_name] = run_or_resume(
        _blend_name, make_blend_runner(_blend_name, _base_name, CONFIGS[_base_name])
    )

print()
print(f"{'config':<32s} {'列数':>5s} {'text列':>6s} {'val(較正後・OOF)':>16s}")
print("-" * 66)
for _name, _r in blend_results.items():
    print(f"{_name:<32s} {int(_r['n_features']):>5d} {int(_r['n_text_features']):>6d} {float(_r['val_seedavg']):>16.6f}")


[2026-08-16 08:10:28] [INFO] ============================================================


INFO:69_catboost_native_text:============================================================


[2026-08-16 08:10:28] [INFO] [R0_memofix_plus_LM_BLEND] 在籍月数回帰+Platt較正+ブレンドを追加（土台: R0_memofix_plus_LM）


INFO:69_catboost_native_text:[R0_memofix_plus_LM_BLEND] 在籍月数回帰+Platt較正+ブレンドを追加（土台: R0_memofix_plus_LM）


[2026-08-16 08:11:04] [INFO]     [回帰] seed=42: 全件学習完了


...

[2026-08-16 08:13:19] [INFO]   最終Platt較正後(OOF)   : 0.508826


INFO:69_catboost_native_text:  最終Platt較正後(OOF)   : 0.508826


[2026-08-16 08:13:19] [INFO]   提出ファイル: 20260816_69_catboost_native_text_R0_plus_LM_plus_TXT_BLEND.csv（予測平均=0.5540）


INFO:69_catboost_native_text:  提出ファイル: 20260816_69_catboost_native_text_R0_plus_LM_plus_TXT_BLEND.csv（予測平均=0.5540）



config                              列数  text列     val(較正後・OOF)
------------------------------------------------------------------
R0_memofix_plus_LM_BLEND           444      0         0.500800
R0_plus_LM_plus_TXT_BLEND          446      2         0.508826


## 15. 結果まとめ

In [29]:
baseline_val = float(standard_results["R0_memofix_plus_LM"]["val_seedavg"])
txt_val = float(standard_results["R0_plus_LM_plus_TXT"]["val_seedavg"])
baseline_blend_val = float(blend_results["R0_memofix_plus_LM_BLEND"]["val_seedavg"])
txt_blend_val = float(blend_results["R0_plus_LM_plus_TXT_BLEND"]["val_seedavg"])

print("=" * 78)
print(f"{'':28s} {'ブレンドなし':>14s} {'+回帰ブレンド+較正':>18s}   {'ブレンドの効果':>14s}")
print(f"{'baseline(444列)':28s} {baseline_val:14.6f} {baseline_blend_val:18.6f}   {baseline_blend_val-baseline_val:+14.6f}")
print(f"{'+TXT(446列)':28s} {txt_val:14.6f} {txt_blend_val:18.6f}   {txt_blend_val-txt_val:+14.6f}")
print("=" * 78)

txt_effect_alone = txt_val - baseline_val
blend_effect_alone = baseline_blend_val - baseline_val
blend_effect_with_txt = txt_blend_val - txt_val
interaction = blend_effect_with_txt - blend_effect_alone
print(f"TXT単体の効果              : {txt_effect_alone:+.6f}")
print(f"回帰ブレンド単体の効果       : {blend_effect_alone:+.6f}"
      "  （第90節の再現値: ブレンド-0.0003・較正+0.0017＝合算+0.0014付近が目安）")
print(f"TXT込みでの回帰ブレンド効果  : {blend_effect_with_txt:+.6f}")
print(f"交互作用（後者-前者）       : {interaction:+.6f}")
print()
if interaction < -0.001:
    print("→ 交互作用はマイナス方向（TXTと組み合わせるとブレンドがより効くようになった）。")
    print("  ユーザーの仮説を支持する結果。")
elif interaction > 0.001:
    print("→ 交互作用はプラス方向（TXTと組み合わせるとブレンドはむしろ効きにくくなった）。")
else:
    print("→ 交互作用はノイズの範囲内（|差|<=0.001）。2つの改善は独立に効いている/効いていないと見るのが妥当。")

rows = []
for name, r in {**standard_results, **blend_results}.items():
    rows.append({"config": name, "列数": int(r["n_features"]), "text列": int(r["n_text_features"]),
                 "val": float(r["val_seedavg"]), "ファイル": Path(r["submission_path"]).name})
summary = pd.DataFrame(rows).set_index("config")

summary["val差(対baseline)"] = summary["val"] - baseline_val
summary["提出"] = "提出する"
for name in ["R0_plus_LM_plus_TXT", "R0_memofix_plus_LM_BLEND", "R0_plus_LM_plus_TXT_BLEND"]:
    if summary.loc[name, "val差(対baseline)"] > VAL_REJECT_MARGIN:
        summary.loc[name, "提出"] = "見送り（足切り）"

pd.set_option("display.width", 220)
print()
print(summary.to_string())
summary.to_csv(OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_summary.csv")
logger.info(f"サマリを保存: {TODAY}_{SCRIPT_NAME}_summary.csv")


                                     ブレンドなし         +回帰ブレンド+較正          ブレンドの効果
baseline(444列)                     0.505477           0.500800        -0.004677
+TXT(446列)                         0.511183           0.508826        -0.002356
TXT単体の効果              : +0.005705
回帰ブレンド単体の効果       : -0.004677  （第90節の再現値: ブレンド-0.0003・較正+0.0017＝合算+0.0014付近が目安）
TXT込みでの回帰ブレンド効果  : -0.002356
交互作用（後者-前者）       : +0.002321

→ 交互作用はプラス方向（TXTと組み合わせるとブレンドはむしろ効きにくくなった）。

                            列数  text列       val                                                            ファイル  val差(対baseline)    提出
config                                                                                                                                
R0_memofix_plus_LM         444      0  0.505477         20260816_69_catboost_native_text_R0_memofix_plus_LM.csv         0.000000  提出する
R0_plus_LM_plus_TXT        446      2  0.511183        20260816_69_catboost_native_text_R0_plus_LM_plus_TXT.csv         0.005705  提出する
R0

INFO:69_catboost_native_text:サマリを保存: 20260816_69_catboost_native_text_summary.csv


## 16. 提出方針

### 判定（事前登録・[[validation-asymmetry]]と同様の考え方）

- **採否は Public のみ。** 検証は足切り（悪化検出）専用
- **足切り**: `VAL_REJECT_MARGIN`（0.02）を超えて悪化したら提出しない（baseline比）
- text_features追加・回帰ブレンド+較正追加は、いずれも「reference/0816版の解析から着想した
  単一の事前登録済み介入」なので、検証が改善を示せばそのまま提出してよい
  （複数構成から検証スコアで選ぶ探索ではない）
- ただし回帰ブレンド+較正は**第90節で既に単体では否定的な結果**（ブレンド-0.0003・較正+0.0017悪化）
  が出ている。ユーザーの仮説（TXTとの相互作用でプラスに転じるか）を`interaction`の値で判定し、
  交互作用が明確にプラスでない限り、4構成のうち`_BLEND`系だけを理由なく採用しない

### 特徴量重要度の確認

`analysis`セクションを追加で走らせる場合、text_featuresの重要度がCatBoostの
`get_feature_importance()`にどう現れるか（テキスト全体で1つの重要度になるか、
内部で自動生成されたBoW/NaiveBayes特徴ごとに分解されるか）を見ておくと、
reference/0816版の「学習・メモ・残業が上位」という報告と比較できる。
